In [1]:
from PIL import Image
import numpy as np
import torch
import os
from torch.utils.data import Dataset , DataLoader

def load_image(path):
    
    img = Image.open(path).convert('RGB')
    img = img.resize((128,128))
    img = np.array(img)
    img = img.astype(np.float32)/255.0
    img = np.transpose(img,(2,0,1))
    img = torch.tensor(img)
    
    return img

img = load_image("/home/ajinkya/leaf.JPG")
print(img.shape)


torch.Size([3, 128, 128])


In [2]:
# create label ("healthy" = 0 , "unhealthy" = 1)

path = "/home/ajinkya/DL_CNN/train"
classes = os.listdir(path)
print(classes)



label_map = {
            "Healthy" : 0,
            "Unhealthy": 1
            }

def load_dataset(folder_path):
    images = []
    labels = []

    for label_name in os.listdir(folder_path):
        
        label_path = os.path.join(folder_path,label_name)

        if not os.path.isdir(label_path):
            continue

        label = label_map[label_name]


        for file in os.listdir(label_path):
            img_path = os.path.join(label_path,file)

            try:
                img = load_image(img_path)
                images.append(img)
                labels.append(label)

            except Exception as e:
                print("Error:", e)
                
    return images, labels


train_images , train_labels = load_dataset("/home/ajinkya/DL_CNN/train")
val_images , val_labels = load_dataset("/home/ajinkya/DL_CNN/val")
test_images , test_labels = load_dataset("/home/ajinkya/DL_CNN/test")

print(" Train : Number of images: " ,len(train_images))
print(" Train : Number of labels: ",len(train_labels))
print("Val : Number of images: " ,len(val_images))
print("Val : Number of labels: ",len(val_labels))
print("Test : Number of images: " ,len(test_images))
print("Test : Number of labels: ",len(test_labels))

print("Shape: " ,train_images[0].shape)
print(label_map)

    

['Healthy', 'Unhealthy']
 Train : Number of images:  1400
 Train : Number of labels:  1400
Val : Number of images:  300
Val : Number of labels:  300
Test : Number of images:  300
Test : Number of labels:  300
Shape:  torch.Size([3, 128, 128])
{'Healthy': 0, 'Unhealthy': 1}


In [4]:
#  Dataset class

class PlantDataset(Dataset):

    def __init__(self,images,labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return(len(self.images))

    def __getitem__(self,idx):
        return self.images[idx],self.labels[idx]


train_dataset = PlantDataset(train_images,train_labels)

img , label = train_dataset[0]
print(img.shape)
print(label)

torch.Size([3, 128, 128])
0


In [5]:
train_dataset = PlantDataset(train_images,train_labels)
val_dataset = PlantDataset(val_images,val_labels)
test_dataset = PlantDataset(test_images,test_labels)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))


train_loader = DataLoader(train_dataset,batch_size=32 , shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=32 , shuffle=False)
test_loader = DataLoader(test_dataset,batch_size=32, shuffle=False)

for images , labels in train_loader:
    print(images.shape)
    print(labels.shape)
    break
    

1400
300
300
torch.Size([32, 3, 128, 128])
torch.Size([32])


## CNN Model

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PlantModel(nn.Module):

    def __init__(self):
        super(PlantModel,self).__init__()
        
        self.layer1 = nn.Conv2d(in_channels = 3 , out_channels= 16 , kernel_size= 3)  #cnn layer1
        self.layer2 = nn.Conv2d(in_channels = 16 , out_channels= 32 , kernel_size= 3) #cnn layer2

        self.fc1 = nn.Linear(32*30*30,128)
        self.fc2 = nn.Linear(128,1)
        
        

    def forward(self,x):

        
        x = F.relu(self.layer1(x)) #conv layer1
        x = F.max_pool2d(x,2)

        
        x = F.relu(self.layer2(x)) #conv layer2
        x = F.max_pool2d(x,2)

       
        x = x.view(x.size(0),-1)   #flatten

        x = F.relu(self.fc1(x))   #Fully connected
        
        x = self.fc2(x)           # Final output
                    

        return x
        

In [7]:
model = PlantModel()
#z = model(images)
print(model)


PlantModel(
  (layer1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))
  (layer2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=28800, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=1, bias=True)
)


In [10]:


loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(8):
    total_loss = 0

    for images , labels in train_loader:
        z = model(images)

        y = labels.float().view(-1,1)
        loss = loss_fn(z,y)
        loss.backward()

        lr = 0.001
        with torch.no_grad():
            for param in model.parameters():
                param -= lr * param.grad 

        for param in model.parameters():
            param.grad = None

        total_loss += loss.item()

    avg_loss = total_loss/len(train_loader)
    print(f"Epoch {epoch}, Loss: { avg_loss:.6f}")

        

    


Epoch 0, Loss: 0.689024
Epoch 1, Loss: 0.688369
Epoch 2, Loss: 0.687728
Epoch 3, Loss: 0.686885
Epoch 4, Loss: 0.686264
Epoch 5, Loss: 0.685484
Epoch 6, Loss: 0.684699
Epoch 7, Loss: 0.683858


In [11]:
def evaluate_acc(loader):
    correct = 0
    total = 0


    with torch.no_grad():
        for images , labels in loader:

            z = model(images)
            p = torch.sigmoid(z)

            preds = (p > 0.5 ).int()
            labels = labels.view(-1,1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct/total

train_accuracy = evaluate_acc(train_loader)
print(f"Training Accuracy: {train_accuracy * 100:.2f}%")



val_accuracy = evaluate_acc(val_loader)
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")


test_accuracy = evaluate_acc(test_loader)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
    

Training Accuracy: 63.21%
Validation Accuracy: 64.67%
Test Accuracy: 63.00%


In [9]:
# save the model 
torch.save(model.state_dict(),"plant_model.pth")


model = PlantModel()
model.load_state_dict(torch.load("plant_model.pth"))
model.eval()


PlantModel(
  (layer1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))
  (layer2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=28800, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=1, bias=True)
)

In [10]:
img = load_image("/home/ajinkya/leaf22.JPG")
img = img.unsqueeze(0)

model.eval()
with torch.no_grad():
    z = model(img)
    p = torch.sigmoid(z)

pred = (p > 0.5).item()

if pred == 0:
    print("Healthy")
else:
    print("Unhealthy")


Healthy
